In [1]:
import string
import re
import pandas as pd
import nltk
import json
from collections import defaultdict, Counter
import json
import os
import pandas as pd
from pathlib import Path

from lxml.html.diff import token



In [2]:
import json
import re
import unicodedata
import datetime

def slugify(value: str) -> str:
    """
    Convert a string to a snake_case ASCII-only identifier.
    """
    value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^\w]+", "_", value.lower()).strip("_")

def build_entity(
    term: str,
    category: str,
    aliases: list,
    regex_pattern: str,
    game_version:str = None,
    canonical_id: str = None,
    introduced_in: str = None,
    deprecated_in: str = None,
    lifecycle_status: str = None,
    topic_area: str = None,
    lang_variant: str = "en-US",
    sentiment_valence: float = None,
    toxicity_risk: str = None,
    frequency_band: str = None,
    longevity_band: str = None,
    player_archetype_bias: dict = None,
    notes: str = None,
    source_count: int = None
) -> dict:
    """
    Build a keyword/entity record following the schema.

    Parameters:
        topic_area:
        lifecycle_status:
        deprecated_in:
        introduced_in:
        game_version:
        term: the display text of the keyword
        category: high-level classification (e.g., "ability", "raid_size")
        aliases: list of alias strings for matching
        regex_pattern: a regex to capture the term in text
        canonical_id: optional override for the slug; defaults to slug(term)
        introduced_in, deprecated_in, lifecycle_status, topic_area: optional metadata
        lang_variant: e.g., "en-US"
        sentiment_valence: optional sentiment score
        toxicity_risk: e.g., "low", "medium", "high"
        frequency_band: optional frequency category
        longevity_band: optional longevity category
        player_archetype_bias: optional dict of archetype likelihoods
        notes: free-text notes
        source_count: how many sources mentioned this term
    """
    cid = canonical_id or slugify(term)
    today = datetime.date.today().isoformat()

    entity = {
        "term": term,
        "canonical_id": cid,
        "category": category,
        "game_version": game_version,
        "aliases": [{"text": a, "readability": None} for a in aliases],
        "introduced_in": introduced_in,
        "deprecated_in": deprecated_in,
        "lifecycle_status": lifecycle_status,
        "topic_area": topic_area,
        "regex_pattern": regex_pattern,
        "lang_variant": lang_variant,
        "sentiment_valence": sentiment_valence,
        "toxicity_risk": toxicity_risk,
        "frequency_band": frequency_band,
        "longevity_band": longevity_band,
        "player_archetype_bias": player_archetype_bias,
        "metadata": {
            "notes": notes,
            "source_count": source_count,
            "last_updated": today,
            "schema_version": "1.1"
        }
    }
    return entity

def save_entities(entities: list, filepath: str):
    """
    Save a list of entity dicts to a JSONL file.
    """
    with open(filepath, "w", encoding="utf-8") as f:
        for e in entities:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

def append_entities(entities: list, filepath: str):
    """
    Append a list of entity dicts to a JSONL file.
    """
    with open(filepath, "a+", encoding="utf-8") as f:
        for e in entities:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

In [3]:
data = pd.read_csv('all_words.csv')

In [4]:
data

,term,category
0,1-point,NaN
1,1-shot,NaN
2,1.1.0,NaN
3,1.10.0,NaN
4,1.11.0,NaN
...,...,...
5893,zek,NaN
5894,elites,NaN
5895,elite,NaN
5896,trinkets,NaN


In [5]:
PUNCT_RE = re.compile(f"[{re.escape(string.punctuation)}]")
data["contains_punct"] = data["term"].apply(lambda s: bool(PUNCT_RE.search(s)))

In [6]:
len(data['term'].unique())

5489

In [7]:
df = data.drop_duplicates(subset='term', keep='first')

In [8]:
df

,term,category,contains_punct
0,1-point,NaN,True
1,1-shot,NaN,True
2,1.1.0,NaN,True
3,1.10.0,NaN,True
4,1.11.0,NaN,True
...,...,...,...
5892,hydra,NaN,False
5893,zek,NaN,False
5895,elite,NaN,False
5896,trinkets,NaN,False


In [9]:
df[df['term'].str.contains(r'\+')]

,term,category,contains_punct
161,alt+f4,NaN,True
1846,m +,NaN,True
1847,m+,NaN,True
2067,mythic+,NaN,True
3616,mythic +,NaN,True
3700,M+,NaN,True
3701,M+ dungeons,NaN,True
3793,classic+,NaN,True
3794,classic +,NaN,True
4931,Mythic+,NaN,True


In [10]:
import re
import json
import pandas as pd

# --- Script: wow_title_search.py ---
# Usage examples are included at the bottom of the script

def load_titles(filepath: str) -> list:
    """Load video titles from a text file, one title per line."""
    with open(filepath, encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

def load_entities(jsonl_path: str) -> list:
    """Load WoW entities (terms + aliases + regex) from a JSONL file."""
    entities = []
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            pattern = re.compile(obj["regex_pattern"], re.IGNORECASE)
            entities.append((obj["term"], [a["text"] for a in obj["aliases"]], pattern))
    return entities

def find_wow_terms_by_entities(titles: list, entities: list) -> pd.DataFrame:
    """
    Scan a list of titles for WoW terms defined by entities.
    Returns a DataFrame with columns: title, wow_terms.
    """
    results = []
    for title in titles:
        found = set()
        for term, aliases, pattern in entities:
            if pattern.search(title):
                found.add(term)
        results.append({"title": title, "wow_terms": sorted(found)})
    return pd.DataFrame(results)

def find_wow_terms_by_vocab(titles: list, vocab: list) -> pd.DataFrame:
    """
    Scan titles for a simple seed vocabulary (exact matches).
    Returns a DataFrame with columns: title, wow_terms.
    """
    patterns = [(term, re.compile(rf"\b{re.escape(term)}\b", re.IGNORECASE)) for term in vocab]
    results = []
    for title in titles:
        found = set()
        for term, pat in patterns:
            if pat.search(title):
                found.add(term)
        results.append({"title": title, "wow_terms": sorted(found)})
    return pd.DataFrame(results)

def extract_candidate_phrases(titles: list) -> pd.DataFrame:
    """
    Extract capitalized or symbol-containing phrases as new candidates.
    Returns a DataFrame of phrase and count occurrences.
    """
    phrase_pattern = re.compile(r"[A-Z][\w+'\+]*(?:\s+[A-Z][\w+'\+]*)*")
    phrases = set()
    for title in titles:
        for match in phrase_pattern.findall(title):
            if len(match) > 2 and match.lower() not in {
                "the","and","for","to","in","on","of","this","that","with"
            }:
                phrases.add(match.strip())
    counts = {p: sum(p in t for t in titles) for p in phrases}
    df = pd.DataFrame([{"phrase": p, "count": counts[p]} for p in counts])
    return df.sort_values("count", ascending=False).reset_index(drop=True)

# -------- Example usage --------
titles = load_titles("unique_video_titles.txt")
entities = load_entities("updated_entities_with_game_version.jsonl")

# 1. Match using your entity definitions
df_entities = find_wow_terms_by_entities(titles, entities)

# 2. Match using a simple seed vocabulary
seed_vocab = [
    "Mythic+", "Dungeon", "Raid", "AFK", "Addon", "Transmog", "Pets",
    "Gold farming", "Torghast", "Conduit", "Renown", "Anima", "Covenant",
    "LFG", "LFR", "PvP", "PvE", "Spec", "Speedrun"
]
df_vocab = find_wow_terms_by_vocab(titles, seed_vocab)

# 3. Extract new candidate phrases for review
df_candidates = extract_candidate_phrases(titles)


In [11]:
df_candidates

,phrase,count
0,WoW,9037
1,War,6351
2,PvP,6078
3,Sha,4044
4,PvP WoW,3862
...,...,...
41431,These Classes Have Some Of The BIGGEST Burst F...,1
41432,Westmarch Achievement Guide,1
41433,Arcane Mages Are RAID BOSSES Of TWW Beta,1
41434,AFFLICTION LOCK POWER,1


In [12]:
df_candidates_copy = pd.read_csv('candidates_copy.csv')

In [13]:
df_candidates_copy

,phrase,count
0,War,6351.0
1,PvP,6078.0
2,Sha,4044.0
3,Shadowland,3303.0
4,Shadowlands,3292.0
...,...,...
39302,Covenant,NaN
39303,Augmentation Evoker,NaN
39304,Al'Akir,NaN
39305,Aldor,NaN


In [14]:
df_candidates_copy.drop_duplicates(subset='phrase', keep='first', inplace=True)


In [15]:
df_candidates_copy['clean']=df_candidates_copy['phrase'].str.lower()

In [16]:
df_candidates_copy.drop_duplicates(subset='clean', keep='first', inplace=True)

In [17]:
df_candidates_copy['tokens'] = df_candidates_copy['clean'].apply(lambda x: x.split())

In [18]:
df_candidates_copy

,phrase,count,clean,tokens
0,War,6351.0,war,[war]
1,PvP,6078.0,pvp,[pvp]
2,Sha,4044.0,sha,[sha]
3,Shadowland,3303.0,shadowland,[shadowland]
4,Shadowlands,3292.0,shadowlands,[shadowlands]
...,...,...,...,...
39294,Zul'Farrak,1.0,zul'farrak,[zul'farrak]
39295,Zul'Gurub,1.0,zul'gurub,[zul'gurub]
39296,Zygor Vs RestedXP,1.0,zygor vs restedxp,"[zygor, vs, restedxp]"
39298,fel pup,NaN,fel pup,"[fel, pup]"


In [19]:
words = " ".join(df_candidates_copy['clean'])

In [20]:
token = nltk.word_tokenize(words)

In [36]:
tokens = [token for token in token if token not in nltk.corpus.stopwords.words('english')]

In [37]:
tokens = list(set(tokens))

In [38]:
tokens

['cleaning',
 'dreamtalon',
 'throwback',
 'tempetation',
 'incarnates',
 'op',
 'ignis',
 'godcomp',
 'marskman',
 'exists',
 'gigachad',
 'combust',
 'zereth',
 'hall',
 'saviour',
 'butchered',
 'sprockets',
 'bus',
 'stomp',
 'download',
 'conversation',
 'professional',
 'loose',
 'international',
 'pens',
 'brf',
 'drkael',
 'sidereal',
 'longest',
 'deadlier',
 'supposed',
 'tasks',
 'wrecking',
 'unranked',
 'become',
 'ready',
 'server',
 'blackrook',
 'naked',
 'lo',
 'nerus',
 'grindy',
 'click',
 'principle',
 'eranog',
 'najka',
 'badasses',
 'lives',
 'audio',
 'wares',
 "t'uure",
 'room',
 'malefic',
 'sooty',
 'responsibility',
 'cheats',
 'theorizing',
 'titans',
 'microtransactions',
 'popularity',
 'kungarna',
 'greatness',
 'razorscale',
 'prey',
 'summit',
 'oldness',
 'vaults',
 'erased',
 'tiger',
 'instead',
 'noth',
 'scarier',
 'kfc',
 'seriously',
 'lovely',
 'citrine',
 'fury',
 'bracket',
 'olive',
 'later',
 'rats',
 'sorted',
 "mort'regar",
 'deleted',
 '

In [39]:
# with open('tokens.txt', 'w') as fp:
#     for t in tokens:
#         fp.write(f"{t}\n")


In [40]:
with open('tokens.txt', 'r') as fp:
    tokens = fp.readlines()

In [41]:
str_tokens = [t.strip() for t in tokens]

In [42]:
str_tokens.sort()

In [43]:
str_tokens

['ab',
 'abathur',
 'abduction',
 'aberrus',
 'abk',
 'absorb',
 'abuse',
 'abusing',
 'abyss',
 'academy',
 'accept',
 'accepted',
 'access',
 'accessible',
 'accessories',
 'accord',
 'account',
 'accusations',
 'ace',
 'achi',
 'achieved',
 'achievement',
 'achievements',
 'achieves',
 'actiblizz',
 'activision',
 'addon',
 'addons',
 'affix',
 'affixes',
 'affliction',
 'aggramar',
 "ahn'qiraj",
 'ak',
 "al'akir",
 'albatross',
 'alchemist',
 'alchemy',
 'aldor',
 'aldrachi',
 'alert',
 'alexandros',
 'alexstrasza',
 'algalon',
 'algar',
 'algethar',
 'alleria',
 'alliance',
 'allied',
 'allies',
 'alluriel',
 'ally',
 'almai',
 'alpaca',
 'alpacas',
 'alt',
 'altar',
 'alterac',
 'altimor',
 'alts',
 'alunira',
 'aluriel',
 "aman'thul",
 'amirdrassil',
 'ams',
 'amz',
 "an'she",
 'anathema',
 'anduin',
 'andybrew',
 'anima',
 'ansurek',
 'antorus',
 "anub'arak",
 'anveena',
 'anzu',
 'aoc',
 'aoe',
 'aotc',
 'ap',
 'ape',
 'aq',
 'aq20',
 'aq40',
 'ar',
 'ara',
 'arak',
 'arakkoa'

In [44]:
str_tokens = list(set(str_tokens))

In [45]:
str_tokens.sort()

In [46]:
str_tokens

['ab',
 'abathur',
 'abduction',
 'aberrus',
 'abk',
 'absorb',
 'abuse',
 'abusing',
 'abyss',
 'academy',
 'accept',
 'accepted',
 'access',
 'accessible',
 'accessories',
 'accord',
 'account',
 'accusations',
 'ace',
 'achi',
 'achieved',
 'achievement',
 'achievements',
 'achieves',
 'actiblizz',
 'activision',
 'addon',
 'addons',
 'affix',
 'affixes',
 'affliction',
 'aggramar',
 "ahn'qiraj",
 'ak',
 "al'akir",
 'albatross',
 'alchemist',
 'alchemy',
 'aldor',
 'aldrachi',
 'alert',
 'alexandros',
 'alexstrasza',
 'algalon',
 'algar',
 'algethar',
 'alleria',
 'alliance',
 'allied',
 'allies',
 'alluriel',
 'ally',
 'almai',
 'alpaca',
 'alpacas',
 'alt',
 'altar',
 'alterac',
 'altimor',
 'alts',
 'alunira',
 'aluriel',
 "aman'thul",
 'amirdrassil',
 'ams',
 'amz',
 "an'she",
 'anathema',
 'anduin',
 'andybrew',
 'anima',
 'ansurek',
 'antorus',
 "anub'arak",
 'anveena',
 'anzu',
 'aoc',
 'aoe',
 'aotc',
 'ap',
 'ape',
 'aq',
 'aq20',
 'aq40',
 'ar',
 'ara',
 'arak',
 'arakkoa'

In [47]:
all_words = data['term'].str.lower()

In [48]:
all_words = set(list(all_words))
set_tokens = set(str_tokens)

In [49]:
intersect_words=all_words.intersection(set_tokens)

In [50]:
intersect_words

{'elvui',
 'pyro',
 'shotted',
 'squid',
 'arakkoa',
 'incarn',
 'talanji',
 'mr',
 'magtheridon',
 'dmg',
 'resurrection',
 'salt',
 'godcomp',
 'class',
 'serenity',
 'shadowplay',
 'brez',
 'gigachad',
 'frostwolf',
 'bwl',
 'ring',
 'green',
 'dungeons',
 'yazma',
 'durotan',
 'gcd',
 'tv',
 'nerfed',
 "mal'ganis",
 'wings',
 'vdh',
 'enh',
 'soothe',
 'cc',
 'enhance',
 'offensive',
 'nuked',
 'm+',
 'rage',
 'paladin',
 'laggy',
 'prog',
 'capped',
 'wow',
 'dht',
 'shammy',
 'twink',
 'gnome',
 'silvermoon',
 'prot',
 'fade',
 'corruption',
 'hunters',
 'map',
 'rng',
 'rap',
 'wep',
 'lag',
 'wall',
 'voti',
 'heals',
 'nagrand',
 'soloing',
 'stacks',
 'duskwood',
 'darkspear',
 'malfurion',
 'bufonid',
 'fmp',
 'grindy',
 'krexus',
 "kel'thuzad",
 'st',
 'warsong',
 'raszageth',
 'scuffed',
 'respec',
 'maul',
 'spriest',
 'blizzcon',
 'azeroth',
 'draenor',
 'toons',
 'mark',
 'prevoker',
 'sethrak',
 'spamming',
 'galakrond',
 'dr',
 'archimonde',
 'guildmates',
 'arc',
 'r

In [51]:
with open('intersect_words.txt', 'w') as fp:
    for t in intersect_words:
        fp.write(f"{t}\n")


In [16]:
with open('all_words_v2.txt', 'r') as fp:
    words = fp.readlines()

In [17]:
len(words)

79

In [18]:
words = [word.strip() for word in words]

In [19]:
words = list(set(words))

In [20]:
len(words)

55

In [23]:
words = sorted(words)

In [24]:
with open('all_words_v2.txt', 'w') as fp:
    for t in words:
        fp.write(f"{t}\n")